## Storage & logging drivers

Two pluggable subsystems set in `daemon.json` — one decides how *image layers* stack on disk, the other where *container output* goes.

**Storage drivers** implement the layered, copy-on-write filesystem from module 02:

| Driver | Backend | Notes |
|---|---|---|
| **`overlay2`** *(default)* | kernel overlayfs | fast, stable — what ~everyone uses |
| `btrfs` / `zfs` | those filesystems | native snapshots; niche |
| `fuse-overlayfs` | overlay via FUSE | rootless on older kernels |
| `vfs` | full copies, no COW | testing only — slow, huge |
| `devicemapper` | LVM | **deprecated** |

The guidance is short: **keep `overlay2`** — check with `docker info | grep -i storage`. Switching drivers is **destructive** (existing images/containers don't migrate), so it's a fresh-install decision, not a live change.

**Logging drivers** decide where each container's stdout/stderr lands. The default **`json-file`** writes to `/var/lib/docker/containers/<id>/…-json.log`, which is what `docker logs` reads. The catch from module 03: **`docker logs` only works with `json-file`, `local`, and `journald`** — switch to `syslog`, `fluentd`, `awslogs`, `gelf`, or `splunk` and the output goes straight to that system, and `docker logs` shows nothing.

Set the driver at three levels: the **daemon default** (`log-driver` in `daemon.json`), **per container** (`--log-driver`/`--log-opt`), or **per Compose service** (`logging:`). The common shape: `json-file` **with rotation** (`max-size`/`max-file`) on dev hosts, and a **centralised** driver (fluentd, awslogs, splunk) shipping to a log aggregator in prod. One gotcha: changing the daemon default **doesn't migrate existing containers** — only ones created afterward pick it up.